# Sentiment–Rating Incongruence in Sri Lanka Tourism Reviews
### Complete Research Analysis Notebook — Version 2

**Dataset:** 16,156 tourism attraction reviews sourced from Mendeley Data (2011–2023)  
**Study:** Measuring, mapping, and explaining the mismatch between star ratings and text sentiment

---
| RQ | Question |
|---|---|
| **RQ1** | How prevalent is sentiment–rating incongruence, and what are ALL directional patterns? |
| **RQ2** | Does incongruence vary significantly across location types and temporal phases? |
| **RQ3** | Which contextual, behavioural, and temporal factors independently predict incongruence? |

---
### The Four Reviewer Types (NEW — all four classified)
| Pattern | Rating Class | Sentiment | Behaviour |
|---|---|---|---|
| **Conservative Rater** | Neutral (3★) | POSITIVE text | Enjoyed it but rated cautiously |
| **Obligatory 5-Star** | Positive (4-5★) | NEUTRAL text | High stars, lukewarm writing |
| **Frustrated Rater** | Positive (4-5★) | NEGATIVE text | Gave stars despite complaints |
| **Harsh Rater** | Neutral (3★) | NEGATIVE text | Average stars, very negative text |

In [ ]:
# ======================================================================
# 0. IMPORTS & CONFIGURATION
# ======================================================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # Remove this line when running in Jupyter
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Colour palette
P = dict(
    main='#4A6FA5',
    acc='#E05C5C',
    neu='#F0A500',
    grn='#2E9E6B',
    purple='#8B5CF6',
    teal='#0D9488'
)

# Plot defaults
plt.rcParams.update({
    'figure.dpi'       : 130,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F9F9F7',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.color'       : '#E5E5E0',
    'grid.linewidth'   : 0.6,
    'font.family'      : 'sans-serif',
    'axes.titlesize'   : 13,
    'axes.titleweight' : '600',
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 10,
    'ytick.labelsize'  : 10,
    'legend.fontsize'  : 10,
})

print('Libraries loaded successfully.')

In [ ]:
# ======================================================================
# 1. LOAD DATASET
# ======================================================================
df = pd.read_csv('Processed_Reviews_with_Sentiment.csv')

print(f'Shape          : {df.shape}')
print(f'Year range     : {df["Travel_Date_Year"].min()} – {df["Travel_Date_Year"].max()}')
print(f'Location types : {df["Location_Type"].nunique()}')
print(f'User countries : {df["User_Country"].nunique()}')
print(f'\nRating_Class value counts:')
print(df['Rating_Class'].value_counts())
print(f'\nSentiment value counts:')
print(df['Sentiment'].value_counts())
df.head(3)

---
## Feature Engineering
Four new variables — no external data, no new models.

In [ ]:
# ======================================================================
# 2. FEATURE ENGINEERING
# ======================================================================

# 2a. Incongruent flag (binary dependent variable)
#     A review is incongruent when Rating_Class and Sentiment disagree.
df['Incongruent'] = (~(
    ((df['Rating_Class'] == 'Positive') & (df['Sentiment'] == 'POSITIVE')) |
    ((df['Rating_Class'] == 'Negative') & (df['Sentiment'] == 'NEGATIVE')) |
    ((df['Rating_Class'] == 'Neutral')  & (df['Sentiment'] == 'NEUTRAL'))
)).astype(int)

# 2b. Four-pattern directional label for incongruent reviews
#     All four theoretically possible mismatch types are classified.
def classify_pattern(row):
    if row['Incongruent'] == 0:
        return 'Congruent'
    rc  = row['Rating_Class']
    sen = row['Sentiment']
    # Conservative Rater: gave neutral stars but wrote positively
    if rc == 'Neutral'  and sen == 'POSITIVE': return 'Conservative Rater'
    # Obligatory 5-Star: gave high stars but wrote neutrally
    if rc == 'Positive' and sen == 'NEUTRAL':  return 'Obligatory 5-Star'
    # Frustrated Rater: gave high stars despite writing negatively
    if rc == 'Positive' and sen == 'NEGATIVE': return 'Frustrated Rater'
    # Harsh Rater: gave neutral stars but wrote very negatively
    if rc == 'Neutral'  and sen == 'NEGATIVE': return 'Harsh Rater'
    # Any remaining mismatches (e.g., Negative rating + Positive text)
    return f'Other ({rc} | {sen})'

df['Incongruence_Pattern'] = df.apply(classify_pattern, axis=1)

# 2c. Reviewer Tier (based on User_Contributions count)
df['Reviewer_Tier'] = pd.cut(
    df['User_Contributions'],
    bins=[0, 5, 20, 100, 100_000],
    labels=['Novice (1-5)', 'Casual (6-20)', 'Active (21-100)', 'Expert (101+)']
)

# 2d. Temporal Era (COVID phases)
df['Era'] = df['Travel_Date_Year'].apply(
    lambda y: 'Pre-COVID (2011-2019)'    if y <= 2019
         else 'During COVID (2020-2021)' if y <= 2021
         else 'Post-COVID (2022-2023)'
)

print('Feature Engineering complete.')
print(f'  Incongruent reviews : {df["Incongruent"].sum():,}  ({df["Incongruent"].mean()*100:.1f}%)')
print(f'\nAll four incongruence pattern counts:')
print(df[df['Incongruent']==1]['Incongruence_Pattern'].value_counts())
print(f'\nReviewer Tier distribution:')
print(df['Reviewer_Tier'].value_counts().sort_index())
print(f'\nEra distribution:')
print(df['Era'].value_counts())

---
## RQ1 — Prevalence and ALL Directional Patterns

This section now reports all four theoretically possible incongruence types,
not just two. This is more complete and more novel for the paper.

In [ ]:
# ── 3a. Overall prevalence ─────────────────────────────────────────────
total     = len(df)
inc_count = df['Incongruent'].sum()
con_count = total - inc_count
inc_rate  = inc_count / total * 100

print('=' * 55)
print(f'  Total reviews      : {total:,}')
print(f'  Congruent          : {con_count:,}  ({con_count/total*100:.1f}%)')
print(f'  Incongruent        : {inc_count:,}  ({inc_rate:.1f}%)')
print('=' * 55)
print('  Approximately 1 in 5 reviews shows a mismatch.')

In [ ]:
# ── 3b. All four directional patterns ─────────────────────────────────
direction = (
    df[df['Incongruent'] == 1]
    .groupby(['Rating_Class', 'Sentiment'])
    .size()
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
)
direction['Pct_of_Incongruent'] = (direction['Count'] / inc_count * 100).round(1)
direction['Pct_of_Total']       = (direction['Count'] / total * 100).round(2)
direction['Pattern']            = direction.apply(
    lambda r: classify_pattern({**r, 'Incongruent': 1}), axis=1
)

print('All incongruence directional patterns:')
print('-' * 70)
print(direction[['Pattern', 'Rating_Class', 'Sentiment',
                 'Count', 'Pct_of_Incongruent', 'Pct_of_Total']].to_string(index=False))
print('-' * 70)
print(f'Total incongruent: {inc_count:,}')

In [ ]:
# ── 3c. Figure 1 — Prevalence doughnut + ALL four pattern bars ────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Doughnut
ax1 = axes[0]
wedges, _ = ax1.pie(
    [con_count, inc_count],
    colors=[P['main'], P['acc']],
    startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
ax1.text(0,  0.08, f'{inc_rate:.1f}%', ha='center', va='center',
         fontsize=22, fontweight='700', color=P['acc'])
ax1.text(0, -0.22, 'incongruent', ha='center', va='center',
         fontsize=11, color='#666')
ax1.legend(wedges, ['Congruent', 'Incongruent'],
           loc='lower center', bbox_to_anchor=(0.5, -0.06), frameon=False)
ax1.set_title('Overall Prevalence\n(n = 16,156)', pad=12)
ax1.set_facecolor('white')

# Right: All four patterns as horizontal bar chart
ax2 = axes[1]

# Colour map for each pattern
pattern_colours = {
    'Conservative Rater' : P['acc'],
    'Obligatory 5-Star'  : P['neu'],
    'Frustrated Rater'   : P['purple'],
    'Harsh Rater'        : P['teal'],
}

dir_plot = direction.sort_values('Count')
bcols = [pattern_colours.get(p, '#888') for p in dir_plot['Pattern']]
bars  = ax2.barh(dir_plot['Pattern'], dir_plot['Count'],
                 color=bcols, height=0.55, edgecolor='white')

for bar, val, pct in zip(bars, dir_plot['Count'], dir_plot['Pct_of_Incongruent']):
    ax2.text(bar.get_width() + 8,
             bar.get_y() + bar.get_height() / 2,
             f'{val:,}  ({pct}% of incongruent)',
             va='center', fontsize=9, color='#444')

ax2.set_xlabel('Number of incongruent reviews')
ax2.set_title('All Four Incongruence Pattern Types\n(% of 3,005 incongruent reviews)', pad=10)
ax2.set_xlim(0, dir_plot['Count'].max() * 1.55)

# Legend
legend_patches = [
    Patch(facecolor=c, label=p)
    for p, c in pattern_colours.items()
]
ax2.legend(handles=legend_patches, frameon=False,
           loc='lower right', fontsize=9)

plt.tight_layout(pad=2.5)
plt.savefig('fig1_rq1_prevalence.png', bbox_inches='tight')
plt.show()

print('\nFour-pattern summary:')
for _, row in direction.iterrows():
    print(f'  {row["Pattern"]:<22}: {row["Count"]:>5,}  ({row["Pct_of_Incongruent"]:>5.1f}% of incongruent)')

---
## RQ2 — Location Type Effect and Temporal Trend

In [ ]:
# ── 4a. Rates and odds ratios by location type ─────────────────────────
lt_stats = df.groupby('Location_Type').agg(
    Total       = ('Incongruent', 'count'),
    Incongruent = ('Incongruent', 'sum'),
    Rate        = ('Incongruent', 'mean')
).sort_values('Rate', ascending=False)

lt_stats['Rate_Pct'] = (lt_stats['Rate'] * 100).round(1)
lt_stats['Not_Inc']  = lt_stats['Total'] - lt_stats['Incongruent']
lt_stats['Odds']     = lt_stats['Incongruent'] / lt_stats['Not_Inc']
ref_odds             = lt_stats.loc['National Parks', 'Odds']
lt_stats['OR']       = (lt_stats['Odds'] / ref_odds).round(2)

# Chi-square test
ct_lt              = pd.crosstab(df['Location_Type'], df['Incongruent'])
chi2, p_lt, dof, _ = chi2_contingency(ct_lt)
n                  = ct_lt.values.sum()
cramers_v          = np.sqrt(chi2 / (n * (min(ct_lt.shape) - 1)))

print('Location Type — Incongruence Rates and Odds Ratios vs National Parks:')
print(lt_stats[['Total', 'Incongruent', 'Rate_Pct', 'OR']].to_string())
print(f'\nChi-square: chi2({dof}) = {chi2:.2f}, p = {p_lt:.6f}')
print(f"Cramer's V = {cramers_v:.4f}  (small but significant effect)")

In [ ]:
# ── 4b. Figure 2 — Location type incongruence bar chart ───────────────
fig, ax = plt.subplots(figsize=(11, 5))

lt_plot = lt_stats.sort_values('Rate_Pct', ascending=True)
bcols   = [P['acc'] if r > inc_rate else P['main'] for r in lt_plot['Rate_Pct']]
bars    = ax.barh(lt_plot.index, lt_plot['Rate_Pct'],
                  color=bcols, height=0.6, edgecolor='white')

for bar, rate, OR in zip(bars, lt_plot['Rate_Pct'], lt_plot['OR']):
    ax.text(bar.get_width() + 0.2,
            bar.get_y() + bar.get_height() / 2,
            f'{rate:.1f}%  (OR={OR}x)',
            va='center', fontsize=9.5, color='#444')

ax.axvline(inc_rate, color='#888', linestyle='--', linewidth=1.3,
           label=f'Overall mean ({inc_rate:.1f}%)')
ax.set_xlabel('Incongruence Rate (%)')
ax.set_title(
    f'Sentiment–Rating Incongruence by Location Type\n'
    f'chi2({dof}) = {chi2:.2f}, p < 0.001, Cramér\'s V = {cramers_v:.3f}',
    pad=10
)
ax.set_xlim(0, lt_plot['Rate_Pct'].max() * 1.35)
ax.legend(frameon=False)

h_above = Patch(facecolor=P['acc'],  label='Above average incongruence')
h_below = Patch(facecolor=P['main'], label='Below average incongruence')
ax.legend(handles=[h_above, h_below], frameon=False, loc='lower right')

plt.tight_layout(pad=2)
plt.savefig('fig2_rq2_location_type.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 5a. Temporal trend: yearly + era ──────────────────────────────────
year_stats = df.groupby('Travel_Date_Year').agg(
    Count       = ('Incongruent', 'count'),
    Incongruent = ('Incongruent', 'sum')
)
year_stats['Rate_Pct'] = (year_stats['Incongruent'] / year_stats['Count'] * 100).round(1)
year_stats['Avg_Rating'] = df.groupby('Travel_Date_Year')['Rating'].mean().round(2)
year_stats['Neg_Sent_Pct'] = (
    df[df['Sentiment'] == 'NEGATIVE'].groupby('Travel_Date_Year').size() /
    df.groupby('Travel_Date_Year').size() * 100
).round(1)

# Spearman trend test
rho, p_rho = spearmanr(year_stats.index, year_stats['Rate_Pct'])
print(f'Spearman rho = {rho:.3f}, p = {p_rho:.4f}')
print('Note: Not monotonic year-on-year, but era-level shift is significant.')

# Era-level chi-square
ct_era              = pd.crosstab(df['Era'], df['Incongruent'])
chi2_e, p_e, _, _  = chi2_contingency(ct_era)

era_rates = (
    df.groupby('Era')['Incongruent']
    .apply(lambda x: round(x.mean()*100, 1))
)
print(f'\nEra chi-square: chi2 = {chi2_e:.2f}, p = {p_e:.4f}')
print('\nIncongruence rate by era:')
print(era_rates)
print(year_stats[['Count', 'Incongruent', 'Rate_Pct']].to_string())

In [ ]:
# ── 5b. Figure 3 — Dual-axis temporal chart ────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

years = year_stats.index.astype(int)

# Era background shading for both panels
for ax in [ax1, ax2]:
    ax.axvspan(2011, 2019.5, alpha=0.04, color=P['main'],  label='Pre-COVID')
    ax.axvspan(2019.5, 2021.5, alpha=0.08, color=P['acc'], label='During COVID')
    ax.axvspan(2021.5, 2024, alpha=0.04, color=P['grn'],   label='Post-COVID')

# Panel A: volume bars + incongruence rate line
ax1b = ax1.twinx()
ax1.bar(years, year_stats['Count'], color=P['main'], alpha=0.35,
        width=0.7, label='Review volume', zorder=2)
ax1b.plot(years, year_stats['Rate_Pct'], color=P['acc'],
          linewidth=2, marker='o', markersize=5,
          label='Incongruence rate (%)', zorder=3)
ax1b.axhline(inc_rate, color='#999', linestyle=':', linewidth=1.2)

ax1.set_ylabel('Review Count', color=P['main'])
ax1b.set_ylabel('Incongruence Rate (%)', color=P['acc'])
ax1.set_title(
    f'Temporal Analysis: Review Volume and Incongruence Rate 2011–2023\n'
    f'Spearman rho={rho:.3f} (p={p_rho:.3f}); Era chi2={chi2_e:.2f} (p={p_e:.4f})',
    pad=8
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1b.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc='upper left')

# Panel B: average rating vs negative sentiment % over time
ax2b = ax2.twinx()
ax2.plot(years, year_stats['Avg_Rating'], color=P['grn'],
         linewidth=2, marker='s', markersize=5, label='Avg Rating')
ax2b.plot(years, year_stats['Neg_Sent_Pct'], color=P['purple'],
          linewidth=2, marker='^', markersize=5, linestyle='--',
          label='Negative Sentiment %')

ax2.set_ylabel('Average Rating', color=P['grn'])
ax2b.set_ylabel('Negative Sentiment (%)', color=P['purple'])
ax2.set_title('Post-COVID Paradox: Ratings Rise While Negative Sentiment Also Rises', pad=6)
ax2.set_xlabel('Year of Travel')
ax2.set_xticks(years)

lines3, labels3 = ax2.get_legend_handles_labels()
lines4, labels4 = ax2b.get_legend_handles_labels()
ax2b.legend(lines3 + lines4, labels3 + labels4, frameon=False, loc='upper left')

# Era labels on top panel
for ax in [ax1]:
    ax.text(2015, ax.get_ylim()[1]*0.9, 'Pre-COVID', color=P['main'],
            fontsize=9, alpha=0.7, ha='center')
    ax.text(2020.5, ax.get_ylim()[1]*0.9, 'COVID', color=P['acc'],
            fontsize=9, alpha=0.7, ha='center')
    ax.text(2022.5, ax.get_ylim()[1]*0.9, 'Post', color=P['grn'],
            fontsize=9, alpha=0.7, ha='center')

plt.tight_layout(pad=2)
plt.savefig('fig3_rq3_temporal.png', bbox_inches='tight')
plt.show()

---
## RQ3 — Predictor Analysis
### Part A: Reviewer Tier and Behavioural Variables

In [ ]:
# ── 6a. Reviewer Tier ──────────────────────────────────────────────────
tier_stats = df.groupby('Reviewer_Tier', observed=True).agg(
    Count       = ('Incongruent', 'count'),
    Incongruent = ('Incongruent', 'sum'),
    Rate        = ('Incongruent', 'mean')
)
tier_stats['Rate_Pct'] = (tier_stats['Rate'] * 100).round(1)
tier_stats['Not_Inc']  = tier_stats['Count'] - tier_stats['Incongruent']
tier_stats['Odds']     = tier_stats['Incongruent'] / tier_stats['Not_Inc']
novice_odds            = tier_stats.loc['Novice (1-5)', 'Odds']
tier_stats['OR']       = (tier_stats['Odds'] / novice_odds).round(2)

ct_tier              = pd.crosstab(df['Reviewer_Tier'], df['Incongruent'])
chi2_ti, p_ti, _, _ = chi2_contingency(ct_tier)

print('Reviewer Tier — Incongruence Rates and Odds Ratios vs Novice:')
print(tier_stats[['Count', 'Incongruent', 'Rate_Pct', 'OR']].to_string())
print(f'\nChi-square: chi2 = {chi2_ti:.2f}, p = {p_ti:.2e}')
print('Expert reviewers (101+ contributions) are ~2x more likely to be incongruent.')

In [ ]:
# ── 6b. Review Length ──────────────────────────────────────────────────
len_inc = df[df['Incongruent'] == 1]['Review_Length']
len_con = df[df['Incongruent'] == 0]['Review_Length']
u_len, p_len = mannwhitneyu(len_inc, len_con, alternative='two-sided')

print('Mann-Whitney U — Review Length (Incongruent vs Congruent)')
print(f'  Median incongruent : {len_inc.median():.0f} chars')
print(f'  Median congruent   : {len_con.median():.0f} chars')
print(f'  U = {u_len:.0f}, p = {p_len:.4f}')
if p_len < 0.05:
    print('  SIGNIFICANT: Incongruent reviews are systematically longer.')
else:
    print('  Not significant.')

In [ ]:
# ── 6c. Review Delay ───────────────────────────────────────────────────
del_inc = df[df['Incongruent'] == 1]['Review_Delay_Days']
del_con = df[df['Incongruent'] == 0]['Review_Delay_Days']
u_del, p_del = mannwhitneyu(del_inc, del_con, alternative='two-sided')

print('Mann-Whitney U — Review Delay (Incongruent vs Congruent)')
print(f'  Median incongruent : {del_inc.median():.0f} days')
print(f'  Median congruent   : {del_con.median():.0f} days')
print(f'  U = {u_del:.0f}, p = {p_del:.4f}')
if p_del >= 0.05:
    print('  NOT significant — posting delay does not predict incongruence.')
    print('  This null result is itself a finding: timing does not drive the mismatch.')

In [ ]:
# ── 6d. Figure 4 — Reviewer tier bars + Review length violin ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel A: Tier bar chart
ax1   = axes[0]
ts    = tier_stats.sort_index()
tier_colours = [P['grn'], P['neu'], P['main'], P['acc']]
bars  = ax1.bar(ts.index.astype(str), ts['Rate_Pct'],
                color=tier_colours, edgecolor='white', width=0.6)

for bar, rate, OR in zip(bars, ts['Rate_Pct'], ts['OR']):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.3,
             f'{rate}%\nOR={OR}x',
             ha='center', va='bottom', fontsize=9.5, fontweight='500')

ax1.axhline(inc_rate, color='#999', linestyle='--', linewidth=1.2,
            label=f'Overall mean ({inc_rate:.1f}%)')
ax1.set_ylim(0, max(ts['Rate_Pct']) * 1.25)
ax1.set_xlabel('Reviewer Tier')
ax1.set_ylabel('Incongruence Rate (%)')
ax1.set_title(
    f'Incongruence by Reviewer Expertise Tier\n'
    f'chi2 = {chi2_ti:.1f}, p < 0.001\n'
    f'(Experts are ~2x more likely to be incongruent than Novices)',
    pad=8
)
ax1.legend(frameon=False)

# Panel B: Review length violin
ax2       = axes[1]
plot_trim = df[df['Review_Length'] <= df['Review_Length'].quantile(0.97)]

parts = ax2.violinplot(
    [plot_trim[plot_trim['Incongruent'] == 0]['Review_Length'],
     plot_trim[plot_trim['Incongruent'] == 1]['Review_Length']],
    positions=[0, 1], showmedians=True, showextrema=False
)

for pc, col in zip(parts['bodies'], [P['main'], P['acc']]):
    pc.set_facecolor(col)
    pc.set_alpha(0.7)
parts['cmedians'].set_color('white')
parts['cmedians'].set_linewidth(2)

ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Congruent', 'Incongruent'])
ax2.set_ylabel('Review Length (characters)')
ax2.set_title(
    f'Review Length: Congruent vs Incongruent\n'
    f'Mann-Whitney p = {p_len:.4f}\n'
    f'(Incongruent reviews are systematically longer)',
    pad=8
)

diff = int(len_inc.median() - len_con.median())
ax2.text(0.5, 0.92, f'Median diff: +{diff} chars',
         ha='center', transform=ax2.transAxes, fontsize=10, color='#555',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f0', edgecolor='#ddd'))

plt.tight_layout(pad=2.5)
plt.savefig('fig4_rq3_behavioral.png', bbox_inches='tight')
plt.show()

---
## RQ3 — Logistic Regression (Full Predictive Model)

**Dependent variable:** Incongruent (0 = congruent, 1 = incongruent)  
**Reference categories:** National Parks (location type), Novice (reviewer tier)  
**All predictors standardised** for comparable coefficients

In [ ]:
# ── 7a. Build model matrix ─────────────────────────────────────────────
df_lr = df.dropna(subset=['Reviewer_Tier']).copy()

# One-hot encode Location Type — National Parks dropped as reference
lt_dummies = pd.get_dummies(df_lr['Location_Type'], prefix='LT')
if 'LT_National Parks' in lt_dummies.columns:
    lt_dummies = lt_dummies.drop(columns=['LT_National Parks'])

# Ordinal encode Reviewer Tier — Novice = 0 (reference)
tier_map = {
    'Novice (1-5)'   : 0,
    'Casual (6-20)'  : 1,
    'Active (21-100)': 2,
    'Expert (101+)'  : 3
}
df_lr['Tier_ord'] = df_lr['Reviewer_Tier'].map(tier_map)

X = pd.concat([
    lt_dummies.reset_index(drop=True),
    df_lr[['Tier_ord', 'Review_Length',
           'Review_Delay_Days', 'Travel_Date_Year']].reset_index(drop=True)
], axis=1)
y = df_lr['Incongruent'].reset_index(drop=True)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Observations : {len(y):,}')
print(f'Predictors   : {X.shape[1]}')
print(f'Outcome rate : {y.mean()*100:.1f}% incongruent')

In [ ]:
# ── 7b. Fit model + Wald-test p-values ────────────────────────────────
lr = LogisticRegression(max_iter=2000, random_state=42, C=1.0)
lr.fit(X_scaled, y)

# Hessian-based standard errors (Wald test)
p_hat = expit(X_scaled @ lr.coef_[0] + lr.intercept_[0])
W     = np.diag(p_hat * (1 - p_hat))
cov   = np.linalg.inv(X_scaled.T @ W @ X_scaled)
se    = np.sqrt(np.diag(cov))
z     = lr.coef_[0] / se
pvals = 2 * (1 - stats.norm.cdf(np.abs(z)))

results = pd.DataFrame({
    'Predictor': X.columns,
    'Coef'     : lr.coef_[0],
    'OR'       : np.exp(lr.coef_[0]),
    'SE'       : se,
    'Z'        : z,
    'p_value'  : pvals,
})
results['Sig'] = results['p_value'].apply(
    lambda p: '***' if p < 0.001 else
              ('**' if p < 0.01 else
              ('*'  if p < 0.05 else '  ns'))
)
results_sorted = results.sort_values('OR', ascending=False)

y_proba = lr.predict_proba(X_scaled)[:, 1]
auc     = roc_auc_score(y, y_proba)

print(f'AUC-ROC : {auc:.4f}\n')
print(results_sorted[['Predictor', 'Coef', 'OR', 'SE', 'Z', 'p_value', 'Sig']]
      .to_string(index=False, float_format='%.4f'))

In [ ]:
# ── 7c. Figure 5 — Odds Ratio forest plot ─────────────────────────────
rs  = results_sorted.reset_index(drop=True)
vn  = rs['Predictor'].str.replace('LT_', '').str.replace('_ord', ' Tier').values
yp  = np.arange(len(rs))

fig, ax = plt.subplots(figsize=(10, 8))

for i, (_, row) in enumerate(rs.iterrows()):
    col = P['acc'] if row['p_value'] < 0.05 else '#AAAAAA'
    OR  = row['OR']
    lo  = np.exp(row['Coef'] - 1.96 * row['SE'])
    hi  = np.exp(row['Coef'] + 1.96 * row['SE'])
    ax.plot([lo, hi], [i, i],          color=col, linewidth=1.5, zorder=2)
    ax.plot([lo, lo], [i-0.1, i+0.1], color=col, linewidth=1.5, zorder=2)
    ax.plot([hi, hi], [i-0.1, i+0.1], color=col, linewidth=1.5, zorder=2)
    ax.scatter(OR, i, color=col, s=70, zorder=3)
    sig = row['Sig'].strip()
    ax.text(max(hi, OR) + 0.01, i,
            f'{OR:.3f}  {sig}',
            va='center', fontsize=8.5,
            color=P['acc'] if row['p_value'] < 0.05 else '#999')

ax.axvline(1.0, color='#888', linestyle='--', linewidth=1.2)
ax.set_yticks(yp)
ax.set_yticklabels(vn, fontsize=9.5)
ax.set_xlabel('Odds Ratio (with 95% CI)')
ax.set_title(
    f'Logistic Regression: Predictors of Sentiment–Rating Incongruence\n'
    f'AUC-ROC = {auc:.3f}  |  n = {len(y):,}  |  '
    f'Ref: National Parks, Novice tier',
    pad=10
)

h_sig = mlines.Line2D([], [], marker='o', color=P['acc'],
                       markersize=8, linestyle='none', label='Significant (p < 0.05)')
h_ns  = mlines.Line2D([], [], marker='o', color='#AAA',
                       markersize=8, linestyle='none', label='Not significant')
h_ref = mlines.Line2D([], [], color='#888', linestyle='--',
                       linewidth=1.5, label='OR = 1.0 (null)')
ax.legend(handles=[h_sig, h_ns, h_ref], frameon=False, loc='lower right')

plt.tight_layout(pad=2)
plt.savefig('fig5_rq3_logistic_regression.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 7d. Figure 6 — ROC Curve ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
fpr, tpr, _ = roc_curve(y, y_proba)
ax.plot(fpr, tpr, color=P['acc'], linewidth=2,
        label=f'ROC curve (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='#bbb', linestyle='--',
        linewidth=1.2, label='Random classifier')
ax.fill_between(fpr, tpr, alpha=0.07, color=P['acc'])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Logistic Regression Model', pad=8)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('fig6_roc_curve.png', bbox_inches='tight')
plt.show()

---
## Full Statistical Results Summary

In [ ]:
# ── 8. Complete results printout ───────────────────────────────────────
print('=' * 65)
print('  STATISTICAL RESULTS — SENTIMENT-RATING INCONGRUENCE STUDY')
print('=' * 65)

print('\n── RQ1: Prevalence ─────────────────────────────────────────────')
print(f'  Overall rate        : {df["Incongruent"].mean()*100:.1f}%  '
      f'({df["Incongruent"].sum():,} / {len(df):,})')
print('  Four directional pattern types:')
for _, row in direction.iterrows():
    print(f'    {row["Pattern"]:<24}: {row["Pct_of_Incongruent"]}% of incongruent reviews')

print('\n── RQ2: Location Type ──────────────────────────────────────────')
print(f'  chi2({dof}) = {chi2:.2f},  p < 0.001,  Cramér\'s V = {cramers_v:.4f}')
print(f'  Highest: Museums        '
      f'{lt_stats.loc["Museums","Rate_Pct"]}%  '
      f'(OR = {lt_stats.loc["Museums","OR"]}x vs National Parks)')
print(f'  Lowest : National Parks '
      f'{lt_stats.loc["National Parks","Rate_Pct"]}%  (reference category)')

print('\n── Temporal Trend ──────────────────────────────────────────────')
print(f'  Spearman rho = {rho:.3f},  p = {p_rho:.4f}')
print(f'  Era chi2 = {chi2_e:.2f},  p = {p_e:.4f}  (significant)')
for era, rate in era_rates.items():
    print(f'  {era}: {rate}%')

print('\n── RQ3: Predictors ──────────────────────────────────────────────')
print(f'  Reviewer Tier chi2 = {chi2_ti:.2f},  p < 0.001')
print(f'  Expert OR vs Novice ≈ {tier_stats.loc["Expert (101+)","OR"]}x')
print(f'  Review Length Mann-Whitney: p = {p_len:.4f}  SIGNIFICANT')
print(f'  Review Delay  Mann-Whitney: p = {p_del:.4f}  NOT significant')
print(f'  Logistic Regression AUC   = {auc:.4f}')
sig_preds = results_sorted[results_sorted['p_value'] < 0.05]['Predictor'].tolist()
ns_preds  = results_sorted[results_sorted['p_value'] >= 0.05]['Predictor'].tolist()
print(f'  Significant predictors : {", ".join(sig_preds)}')
print(f'  Not significant        : {", ".join(ns_preds)}')
print('=' * 65)

In [ ]:
# ── 9. Verify all figures saved ────────────────────────────────────────
import os
figures = [
    'fig1_rq1_prevalence.png',
    'fig2_rq2_location_type.png',
    'fig3_rq3_temporal.png',
    'fig4_rq3_behavioral.png',
    'fig5_rq3_logistic_regression.png',
    'fig6_roc_curve.png',
]
print('Output figures:')
for f in figures:
    status = 'SAVED  ✓' if os.path.exists(f) else 'MISSING ✗'
    print(f'  [{status}]  {f}')